# Очистка данных о звонках (Calls)

In [38]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import re

import help_130625_dam as h

# Пути к данным
RAW_DATA_DIR    = os.path.join('..', 'Sources')
CLEANED_DIR     = os.path.join('..', 'data', 'cleaned')

CALLS_INPUT  = os.path.join(RAW_DATA_DIR, 'Calls (Done).xlsx')
CALLS_OUTPUT = os.path.join(CLEANED_DIR, 'calls_clean.pkl')

MAPPING_INPUT = os.path.join(CLEANED_DIR, 'contacts_mapping.pkl')
# BUYERS_INFO     = os.path.join(CLEANED_DIR, 'buyers_info.pkl')

# DEALS_CLEAN     = os.path.join(CLEANED_DIR, 'deals_clean.pkl')
# CALLS_CLEAN     = os.path.join(CLEANED_DIR, 'calls_clean.pkl')

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Загрузка данных

In [18]:
# Читаем ID как строки, это предотвращает округление 19-значных чисел при загрузке
df = pd.read_excel(CALLS_INPUT, dtype={'Id': str, 'CONTACTID': str})

df.columns = [h.to_snake(c) for c in df.columns]

n_before = len(df)

print(f'Форма: {df.shape}')

h.descr_df(df, include='all', show_stats=False, show_sample_rows=False)

Форма: (95874, 11)


Форма: (95874, 11)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений
0,id,str,95874,0,95874
1,call_start_time,str,95874,0,68445
2,call_owner_name,str,95874,0,33
3,contactid,str,91941,3933,15214
4,call_type,str,95874,0,3
5,call_duration_in_seconds,float64,95791,83,2619
6,call_status,str,95874,0,11
7,dialled_number,float64,0,95874,0
8,outgoing_call_status,str,86875,8999,4
9,scheduled_in_crm,float64,86875,8999,2


In [20]:
# Содержательные дубликаты (одно время, один менеджер, один контакт и одна длительность)
# cols_to_check охватывает ситуацию, когда CRM создала две записи на одно действие
cols_to_check = ['call_start_time', 'call_owner_name', 'contactid', 'call_duration_in_seconds']
total_involved = df.duplicated(subset=cols_to_check, keep=False).sum()
print(f"Число бизнес-дубликатов: {total_involved}")

Число бизнес-дубликатов: 6416


In [21]:
unique_to_delete = df.duplicated(subset=cols_to_check, keep='first').sum()

print(f"Всего строк, вовлеченных в дублирование: {total_involved}")
print(f"Будет удалено уникальных копий: {unique_to_delete}")

if unique_to_delete > 0:
    df = df.drop_duplicates(subset=cols_to_check, keep='first')
    print(f"Очистка завершена. Осталось строк: {len(df)}")

Всего строк, вовлеченных в дублирование: 6416
Будет удалено уникальных копий: 3275
Очистка завершена. Осталось строк: 92599


На основе предварительного анализа мы планируем удалить:
1. **`outgoing_call_status`**: Для Outbound звонков значения дублируют `call_status` или не несут дополнительной аналитической ценности.
2. **`scheduled_in_crm`**: Поле содержит техническую информацию о том, был ли звонок запланирован. Запланировано 134 звонка из 95874.
3. **`tag`**: Колонка практически не заполнена.
4. **`dialled_number`**: олонка практически не заполнена.

## Исправление типов

In [ ]:
df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')
df['contactid'] = pd.array([h.str_to_int64(v) for v in df['contactid']], dtype='Int64')

# Исправление дат
date_cols = [col for col in df.columns if 'time' in col or 'date' in col]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# Базовая очистка и оптимизация типов
df['call_duration_in_seconds'] = df['call_duration_in_seconds'].fillna(0).astype('int32')

# Текстовые столбцы переводим в category
str_cols = ['call_owner_name', 'call_type', 'call_status']
for col in str_cols:
    df[col] = df[col].astype('category')

# Удаление неиспользуемых столбцов
cols_to_drop = ['dialled_number', 'tag', 'outgoing_call_status', 'scheduled_in_crm']
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f'Количество звонков с неизвестным contactid (тип {df["contactid"].dtype}): {df["contactid"].isna().sum()}')

Количество звонков с неизвестным contactid (тип Int64): 3799


### Пересекающиеся звонки

In [26]:
# Проверка на перекрывающиеся звонки у одного менеджера
# Рассчитаем время окончания звонка
df['call_end_time'] = df['call_start_time'] + pd.to_timedelta(df['call_duration_in_seconds'], unit='s')

# Сортируем для проверки перекрытий
df_sorted = df.sort_values(['call_owner_name', 'call_start_time'])

# Сдвигаем время окончания предыдущего звонка того же менеджера
df_sorted['prev_call_end'] = df_sorted.groupby('call_owner_name')['call_end_time'].shift(1)

# Условие перекрытия
overlapping = df_sorted[df_sorted['call_start_time'] < df_sorted['prev_call_end']].copy()

if len(overlapping) > 0:
    print(f"Обнаружено {len(overlapping)} строк с пересечением времени.")
    overlapping['overlap_seconds'] = (overlapping['prev_call_end'] - overlapping['call_start_time']).dt.total_seconds()
    print("\nРаспределение величины пересечения (секунды):")
    print(overlapping['overlap_seconds'].describe())
else:
    print("Пересекающихся звонков не обнаружено.")


Обнаружено 7913 строк с пересечением времени.

Распределение величины пересечения (секунды):
count   7913.00
mean      72.76
std      269.26
min        1.00
25%        5.00
50%        7.00
75%       14.00
max     7140.00
Name: overlap_seconds, dtype: float64


### Выводы по пересекающимся звонкам:
В данных обнаружено **9107** случаев временного перекрытия звонков у одного и того же менеджера.

**Возможные причины:**
1. **Технические особенности CRM (75% случаев):** Большинство накладок составляют менее 13 секунд. Это может быть связано с тем, что система начинает запись нового звонка или автодозвон до того, как менеджер закроет карточку предыдущего клиента.
2. **Параллельные линии:** Использование гарнитур с поддержкой нескольких вызовов или работа в нескольких вкладках CRM одновременно.
3. Система может инициировать звонок заранее, чтобы минимизировать простой менеджера.
4. **Ошибки логирования данных:** Неточное фиксирование времени завершения (`Call End Time`) при обрыве связи или программных сбоях.
5. **Аномалии (длинные пересечения):** Одиночные случаи накладок в несколько десятков минут могут указывать на "зависшие" сессии звонков, которые не были корректно завершены в системе.

*Данные аномалии не критичны для общего анализа воронки, так как составляют менее 10% данных и в большинстве своем являются короткими техническими накладками.*


### Обогащение данных

In [27]:
# Флаг успешного дозвона: статус «в трубку взяли» + есть длительность
df['is_successful'] = (
    df['call_status'].isin(['Attended Dialled', 'Received']) &
    (df['call_duration_in_seconds'] > 5)
)
print(f'is_successful: {df["is_successful"].sum():,} успешных из {len(df):,} ({df["is_successful"].mean()*100:.1f}%)')

is_successful: 57,552 успешных из 92,599 (62.2%)


In [40]:
# Применяем маппинг дублей контактов (сформирован в 01_cleaning_contacts)
if os.path.exists(MAPPING_INPUT):
    contact_mapping = pd.read_pickle(MAPPING_INPUT)
    affected = df['contactid'].isin(contact_mapping.keys()).sum()
    df['contactid'] = df['contactid'].replace(contact_mapping.to_dict())
    print(f"Маппинг контактов применён: {affected} звонков перепривязаны к мастер-контактам.")
else:
    print("Файл contacts_mapping.pkl не найден. Сначала выполните 01_cleaning_contacts.")

Маппинг контактов применён: 0 звонков перепривязаны к мастер-контактам.


### Итоговый осмотр

In [41]:
h.descr_df(df, include='all', show_stats=True, show_sample_rows=True, show_quartiles=True)

,Тип,Заполнено,Пропуски,% Пропусков,Уникальных,Пример 1,Пример 2,Пример 3,Min,Mean,Median,Max,Range,Q1,Q3,IQR
Признак,,,,,,,,,,,,,,,,
id,Int64,92599,0,0.00,92599,5805028000000805001,5805028000000768006,5805028000000764027,5805028000000764027,5805028000031551488.00,5805028000032459776.00,5805028000056912329,56147968.00,5805028000018547712.00,5805028000045452288.00,26904576.00
call_start_time,datetime64[us],92599,0,0.00,68445,2023-06-30 08:43:00,2023-06-30 08:46:00,2023-06-30 08:59:00,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
call_owner_name,category,92599,0,0.00,33,John Doe,John Doe,John Doe,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
contactid,Int64,88800,3799,4.10,15194,<NA>,<NA>,<NA>,5805028000000645014,5805028000026371072.00,5805028000025419776.00,5805028000056892055,56247296.00,5805028000012352512.00,5805028000039890944.00,27538432.00
call_type,category,92599,0,0.00,3,Inbound,Outbound,Outbound,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
call_duration_in_seconds,int32,92599,0,0.00,2619,171,28,24,0,170.22,9.00,7625,7625.00,4.00,107.00,103.00
call_status,category,92599,0,0.00,11,Received,Attended Dialled,Attended Dialled,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
call_end_time,datetime64[us],92599,0,0.00,90604,2023-06-30 08:45:51,2023-06-30 08:46:28,2023-06-30 08:59:24,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
is_successful,bool,92599,0,0.00,2,True,True,True,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>


### Сохранение

In [44]:
os.makedirs(os.path.dirname(CALLS_OUTPUT), exist_ok=True)
df.to_pickle(CALLS_OUTPUT)
df.to_excel(CALLS_OUTPUT.replace('.pkl', '.xlsx'))

summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Уникальных звонков (id)',
        'Диапазон времени звонков',
        'Уникальных менеджеров',
        'Кол-во неизвестных контактов',
        'Пропуски в финальном DF'
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        df['id'].nunique(),
        f'{df["call_start_time"].min().date()} → {df["call_start_time"].max().date()}',
        df['call_owner_name'].nunique(),
        df['contactid'].isna().sum(),
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {CALLS_OUTPUT}')
display(pd.DataFrame(summary_data))

Сохранено: ../data/cleaned/calls_clean.pkl


,Метрика,Значение
0,Строк исходно,95874
1,Строк после очистки,92599
2,Удалено дубликатов,3275
3,Уникальных звонков (id),92599
4,Диапазон времени звонков,2023-06-30 → 2024-06-21
5,Уникальных менеджеров,33
6,Кол-во неизвестных контактов,3799
7,Пропуски в финальном DF,3799


## Описание датасета

**Источник:** `Calls (Done).xlsx` — выгрузка данных о звонках из CRM  
**Назначение:** анализ активности менеджеров, оценка качества обработки лидов и расчет метрик дозвона.

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID записи о звонке |
| `contactid` | `int64` | ID связанного контакта (связь с `contacts.id`). При загрузке указан `dtype=str`, что сохранило точность 19-значных ID. |
| `call_start_time` | `datetime` | Дата и время начала звонка |
| `call_duration_in_seconds` | `int32` | Длительность разговора в секундах |
| `is_successful` | `bool` | **Флаг дозвона:** True, если длительность > 0 и статус 'Attended Dialled' или 'Received' |
| `call_owner_name` | `category` | Менеджер, совершивший или принявший звонок |
| `call_type` | `category` | Тип звонка (Inbound/Outbound) |
| `call_status` | `category` | Результат (Completed, Missed и др.) |

**Ключевые связи:**
- `contactid` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `call_start_time` → используется для расчета Speed to Lead в общих отчетах.
- `is_successful` → основной фильтр для оценки эффективности коммуникаций.

## Выводы

В данных звонков зафиксированы **6 416 полных дублей** и **9 107 случаев пересечения времени звонков** у одного менеджера (~10% всех записей). Дубли — типичная проблема CRM-логирования: при обрыве соединения или программном сбое система создаёт вторую запись того же звонка. Пересечения возникают из-за автодозвона — новый вызов инициируется до того, как менеджер закрыл карточку предыдущего клиента. Большинство пересечений (75%) длятся менее 13 секунд.

Без удаления дублей показатель звонковой активности менеджеров завышен, SLA рассчитывается по некорректным меткам времени, а метрика дозвона становится ненадёжной. Дополнительно обнаружено **33 уникальных менеджера**, тогда как в `contacts` их только 27: шестеро либо работают только на холодных звонках и не имеют CRM-карточки, либо это технические/деактивированные аккаунты — в обоих случаях их конверсия не отслеживается.

**Что сделано:** полные дубли удалены; 
    технические пересечения оставлены как несущественные для воронки и SLA-расчётов; 
    добавлен флаг `is_successful` (звонок принят + длительность > 0); 
    применён `contacts_mapping.pkl` для перепривязки звонков к мастер-контактам из `01_cleaning_contacts`.

> **Системная рекомендация:** выяснить статус 6 «лишних» менеджеров — активных сотрудников добавить в справочник контактов, служебные аккаунты автодозвона исключить из аналитики эффективности команды продаж.